# HIP-3 Microstructure Analysis

Interactive research notebook for cross-venue market quality analysis.

Covers:
- Realized volatility decomposition
- Funding rate regime identification
- Basis decomposition (oracle tracking error vs. perp premium)
- Kyle's lambda estimation
- Roll implied spread comparison

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.hyperliquid import HyperliquidClient
from src.research.microstructure import (
    realized_volatility,
    roll_spread,
    amihud_illiquidity,
    kyle_lambda,
    basis_decomposition,
    summary_statistics,
)

plt.style.use('dark_background')
sns.set_palette('muted')

hl = HyperliquidClient(cache_ttl_hours=12)

## 1. Data Acquisition

In [ ]:
import time

coins = ['BTC', 'ETH', 'SOL', 'HYPE', 'TURBO']
start_ms = int((time.time() - 90 * 86400) * 1000)

data = {}
for coin in coins:
    candles = hl.get_candles(coin, interval='1h', start_ms=start_ms)
    funding = hl.get_funding_history(coin, start_ms=start_ms)
    if not funding.empty:
        candles = candles.join(funding[['fundingRate', 'premium']], how='left')
        candles['fundingRate'] = candles['fundingRate'].ffill().fillna(0)
        candles['premium'] = candles['premium'].ffill().fillna(0)
    data[coin] = candles
    print(f'{coin}: {len(candles)} bars')

## 2. Realized Volatility Decomposition

In [ ]:
fig, axes = plt.subplots(len(coins), 1, figsize=(14, 3 * len(coins)), sharex=True)

for ax, coin in zip(axes, coins):
    df = data[coin]
    returns = df['close'].pct_change().dropna()
    
    # Rolling 24h realized vol, annualized
    rolling_vol = returns.rolling(24).std() * np.sqrt(365.25 * 24)
    
    ax.plot(rolling_vol.index, rolling_vol.values, linewidth=0.8)
    ax.set_ylabel('Ann. Vol')
    ax.set_title(f'{coin} - Rolling 24h Realized Volatility')
    ax.axhline(y=rolling_vol.median(), color='white', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

## 3. Funding Rate Regime Analysis

In [ ]:
funding_summary = []

for coin in coins:
    df = data[coin]
    if 'fundingRate' not in df.columns:
        continue
    
    fr = df['fundingRate']
    annualized = fr.mean() * 3 * 365.25
    
    funding_summary.append({
        'symbol': coin,
        'mean_bps': fr.mean() * 10000,
        'std_bps': fr.std() * 10000,
        'annualized': annualized,
        'pct_positive': (fr > 0).mean(),
        'max_abs_bps': fr.abs().max() * 10000,
    })

funding_df = pd.DataFrame(funding_summary).set_index('symbol')
funding_df.style.format({
    'mean_bps': '{:.4f}',
    'std_bps': '{:.4f}',
    'annualized': '{:.1%}',
    'pct_positive': '{:.1%}',
    'max_abs_bps': '{:.2f}',
})

## 4. Basis Decomposition

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

for coin in ['BTC', 'ETH', 'SOL']:
    df = data[coin]
    if 'premium' not in df.columns:
        continue
    
    premium_bps = df['premium'] * 10000
    
    axes[0].plot(premium_bps.index, premium_bps.values, label=coin, alpha=0.7, linewidth=0.6)
    
    # Distribution
    axes[1].hist(premium_bps.dropna(), bins=100, alpha=0.5, label=coin, density=True)

axes[0].set_title('Mark-Oracle Premium (bps)')
axes[0].legend()
axes[0].axhline(y=0, color='white', alpha=0.3)

axes[1].set_title('Premium Distribution')
axes[1].set_xlabel('Premium (bps)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Market Quality Metrics

In [ ]:
quality_metrics = []

for coin in coins:
    df = data[coin]
    returns = df['close'].pct_change().dropna()
    
    rvol = realized_volatility(returns, annualize=365.25 * 24)
    roll = roll_spread(returns)
    
    vol_usd = df['volume'] * df['close'] if 'volume' in df.columns else None
    amihud = amihud_illiquidity(returns, vol_usd.iloc[1:]) if vol_usd is not None else np.nan
    
    quality_metrics.append({
        'symbol': coin,
        'annualized_vol': rvol,
        'roll_spread_bps': roll * 10000,
        'amihud': amihud,
        'bars': len(df),
    })

quality_df = pd.DataFrame(quality_metrics).set_index('symbol')
quality_df.style.format({
    'annualized_vol': '{:.1%}',
    'roll_spread_bps': '{:.2f}',
    'amihud': '{:.2e}',
    'bars': '{:,}',
})

## 6. Cross-Asset Correlation Matrix

In [ ]:
# Build return matrix
returns_dict = {}
for coin in coins:
    returns_dict[coin] = data[coin]['close'].pct_change().dropna()

returns_matrix = pd.DataFrame(returns_dict).dropna()
corr = returns_matrix.corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    square=True, ax=ax,
    linewidths=0.5, linecolor='#333'
)
ax.set_title('Cross-Asset Return Correlation (Hourly)')
plt.tight_layout()
plt.show()